# λ_d Phase 2 — Colab Training

12-layer LDStack, D=896, K=4 on wikitext-103.

**Before run:** Runtime → Change runtime type → **T4 GPU**

**To save mid-training:** Runtime → Interrupt execution → auto-saves checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/lambda_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints → {CKPT_DIR}')

In [ ]:
import os, sys
REPO = '/content/EVA-Ai'
if not os.path.exists(REPO):
    !git clone https://github.com/BlackCatSpb/EVA-Ai.git {REPO}
sys.path.insert(0, REPO)
%cd {REPO}
!git pull

In [ ]:
# Install + prepare data (streaming → low RAM)
!pip install -q datasets tokenizers

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

SEQ_LEN = 128
N_CHUNKS = 80000
CACHE = 'wikitext_chunks.npy'

if not os.path.exists(CACHE):
    ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1',
                      split='train', streaming=True)
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
    all_tokens = []
    for i, ex in enumerate(ds):
        all_tokens.extend(tokenizer(ex['text'])['input_ids'])
        if len(all_tokens) >= N_CHUNKS * SEQ_LEN:
            break
        if i % 10000 == 0:
            print(f'  {i} examples, {len(all_tokens)} tokens', flush=True)
    n = len(all_tokens) // SEQ_LEN
    arr = np.array(all_tokens[:n*SEQ_LEN], dtype=np.int32).reshape(n, SEQ_LEN)
    np.save(CACHE, arr)
    print(f'Saved {arr.shape[0]} chunks')
else:
    arr = np.load(CACHE)
    print(f'Cached {arr.shape[0]} chunks')
print(f'Total tokens: {arr.shape[0] * SEQ_LEN:,}')

In [ ]:
print(f'Checkpoints → {CKPT_DIR}')
print('Interrupt cell (Runtime → Interrupt) to save + exit')

!python colab_train.py \
    --ckpt_dir "{CKPT_DIR}" \
    --batch_size 8 \
    --n_train 50000

## Resume after crash/interrupt

1. Re-run cell 1 (mount Drive)
2. Re-run cell 2 (clone repo)
3. Re-run cell 4 (training) — auto-finds latest checkpoint on Drive

Autosave every 1000 steps + on interrupt + at epoch end.